# RT-DETRv2 R18VD — From Scratch Egitim (Google Colab)

Restratified termal dataset ile RT-DETRv2'yi **sifirdan (random init)** egitir.
Kod repo'daki `models/rtdetr_v2_r18vd/` modullerini aynen kullanir; bu notebook
sadece Colab'a uyarlanmis launcher'dir (`run_training.py`'nin Colab karsiligi).

## Baslamadan once (tek seferlik hazirlik)

1. Lokalde dataset'i zip'le:
   ```bash
   cd dataset && zip -r restratified_coco.zip restratified_coco
   ```
2. `restratified_coco.zip`'i Google Drive'ina yukle (varsayilan beklenen yol:
   `MyDrive/restratified_coco.zip` — farkliysa asagida `DRIVE_ZIP`'i degistir).
3. Runtime > Change runtime type > **GPU** sec (T4 yeterli, A100 varsa batch buyut).

## Kesinti / resume

Checkpoint'ler ve `best.pt` dogrudan **Drive'a** yazilir (`MyDrive/runs_new/...`).
Colab oturumu koptugunda: notebook'u bastan calistir, config hucresinde
`RESUME_FROM`'a Drive'daki son `checkpoint-N` yolunu ver, egitim kaldigi
yerden (epoch/step sayaci dogru sekilde) devam eder.

In [1]:
# GPU kontrolu
!nvidia-smi

Tue Jul 14 16:10:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Bagimliliklar. Colab'in kendi torch'u kullanilir (yeniden kurma!).
# transformers >= 4.48 gerekli (RTDetrV2 mimarisi icin).
!pip install -q -U transformers accelerate albumentations torchmetrics pycocotools openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 153.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.1 MB/s eta 0:00:00


In [3]:
# Drive mount: dataset zip'i buradan okunur, run ciktilari buraya yazilir.
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# Repo'yu klonla (dataset.py, train_rtdetr.py, map_evaluator.py vs. buradan gelir).
import os

REPO_URL = "https://github.com/Yukseltt/CNN-models-optimized-to-run-on-edge-devices.git"
REPO_DIR = "/content/CNN-models-optimized-to-run-on-edge-devices"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print("Repo zaten var, guncelleniyor...")
    !git -C {REPO_DIR} pull

Cloning into '/content/CNN-models-optimized-to-run-on-edge-devices'...
remote: Enumerating objects: 1892, done.
remote: Counting objects: 100% (1892/1892), done.
remote: Compressing objects: 100% (1771/1771), done.
remote: Total 1892 (delta 108), reused 1831 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (1892/1892), 180.65 MiB | 16.24 MiB/s, done.
Resolving deltas: 100% (108/108), done.
Updating files: 100% (3022/3022), done.


In [5]:
# Dataset'i Drive'daki zip'ten LOKAL diske ac.
# ONEMLI: Drive uzerinden dogrudan okumak (35k kucuk dosya) cok yavas —
# her zaman /content altina acip oradan oku.
import os

DRIVE_ZIP = "/content/drive/MyDrive/restratified_coco.zip"  # zip'in Drive'daki yolu
DATA_DIR  = "/content/dataset/restratified_coco"

if os.path.exists(os.path.join(DATA_DIR, "train", "_annotations.coco.json")):
    print("Dataset zaten acilmis, atlaniyor.")
else:
    assert os.path.exists(DRIVE_ZIP), f"Zip bulunamadi: {DRIVE_ZIP} — Drive'a yukledin mi?"
    os.makedirs("/content/dataset", exist_ok=True)
    print("Zip aciliyor (2.2 GB, birkac dakika surer)...")
    !unzip -q {DRIVE_ZIP} -d /content/dataset

# Dogrulama: train/val/test + annotation dosyalari yerinde mi?
for split in ("train", "val", "test"):
    ann = os.path.join(DATA_DIR, split, "_annotations.coco.json")
    n   = len(os.listdir(os.path.join(DATA_DIR, split, "images")))
    assert os.path.exists(ann), f"Eksik: {ann}"
    print(f"{split}: {n} goruntu")

Zip aciliyor (2.2 GB, birkac dakika surer)...
train: 35190 goruntu
val: 4470 goruntu
test: 4480 goruntu


In [ ]:
# --- KONFIGURASYON (run_training.py FROM_SCRATCH_CFG'nin Colab uyarlamasi) ---
import sys
from pathlib import Path

# Sadece rtdetr modul dizinini path'e ekle.
# REPO KOKUNU EKLEME: yerel klasorler HF 'datasets' paketini golgeleyebilir
# (bkz. edge_perf/PROJECT.md tuzak #7).
sys.path.insert(0, f"{REPO_DIR}/models/rtdetr_v2_r18vd")

# Run ciktilari (best.pt, last.pt, xlsx, checkpoints) dogrudan Drive'a yazilir
# ki oturum koptugunda kaybolmasin.
RUNS_NEW = "/content/drive/MyDrive/runs_new"

# --- RESUME ---
# Kesilen run'in dizin ADI (None -> fresh egitim baslar).
# 14 Tem A100 run'i epoch 23'te Drive FUSE hatasiyla kesildi -> "..._colab6".
# En yuksek step'li checkpoint otomatik secilir.
RESUME_RUN  = "rtdetr_v2_r18vd_from_scratch_restratified_colab6"
RESUME_FROM = None
if RESUME_RUN is not None:
    _ckpts = sorted(
        Path(f"{RUNS_NEW}/{RESUME_RUN}/checkpoints").glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[1]),
    )
    assert _ckpts, f"Checkpoint bulunamadi: {RUNS_NEW}/{RESUME_RUN}/checkpoints"
    RESUME_FROM = str(_ckpts[-1])

# A100 40GB config (T4 config'inden farklar):
#   PER_DEVICE_TRAIN_BS 16 -> 32  (VRAM bol; LR lineer olceklendi: 2e-4 -> 4e-4)
#   PER_DEVICE_EVAL_BS  16 -> 32  (Hungarian matcher VRAM spike A100'de sorun degil)
#   WORKERS 2 -> 8                (A100 Colab ~12 vCPU)
#   PREFETCH_FACTOR 2 -> 4
#   EVAL_STEPS 2000 -> 1100       (bs32'de ~1100 step = 1 epoch -> her epoch eval+checkpoint)
#   bf16 otomatik secilir (A100 compute 8.0), tf32 aktif — ikisi de train_rtdetr.py'de otomatik.
# NOT: Resume ederken batch/LR degistirme — checkpoint'teki optimizer/scheduler
# state eski LR'a gore. Yeni bs/LR sadece FRESH run icin.
CFG = {
    "EPOCHS":              300,
    "IMAGE_SIZE":          480,
    "PER_DEVICE_TRAIN_BS": 32,
    "PER_DEVICE_EVAL_BS":  32,
    "GRAD_ACCUM":          1,
    "LR":                  4e-4,
    "WARMUP_STEPS":        1500,  # bs32'de ~1.4 epoch warmup — from-scratch icin iyi
    "WEIGHT_DECAY":        1e-4,
    "PATIENCE":            30,
    "WORKERS":             8,
    "PREFETCH_FACTOR":     4,
    "EVAL_ACCUM_STEPS":    4,
    "EVAL_STEPS":          1100,
    "COMPILE":             False, # RT-DETRv2 encoder'i ile uyumsuz (train_rtdetr.py notu)
    "PROJECT":             RUNS_NEW,
    "NAME":                "rtdetr_v2_r18vd_from_scratch_restratified_colab",
    "RESUME_FROM":         None,
    "FROM_SCRATCH":        True,  # random init — pretrain YOK
    "SKIP_CONFIRM":        True,
}

if RESUME_FROM is not None:
    # train_rtdetr: RESUME_FROM ve FROM_SCRATCH ayni anda olamaz.
    # Resume'da run_dir checkpoint yolundan otomatik bulunur (colab6'ya devam),
    # xlsx offset'i son satirdan okunur, best.pt metrigi diskten yuklenir.
    CFG["RESUME_FROM"]  = RESUME_FROM
    CFG["FROM_SCRATCH"] = False
    CFG["WARMUP_STEPS"] = 0
    print(f"RESUME modu: {RESUME_FROM}")
    print(f"Run ciktilari: {RUNS_NEW}/{RESUME_RUN}")
else:
    print("FRESH from-scratch egitim.")
    print(f"Run ciktilari: {RUNS_NEW}/{CFG['NAME']}")

In [ ]:
# --- EGITIM ---
# A100: bf16 + tf32 otomatik aktif (train_rtdetr.py gate'liyor).
# Her EVAL_STEPS'te: eval + Excel satiri + best/last.pt + HF checkpoint (Drive'a).
# Excel/pt yazimlari Drive FUSE hatalarina dayanikli (retry + skip) — egitim olmez.
from train_rtdetr import train

train(data_dir=DATA_DIR, cfg=CFG)

## Egitim sonrasi

Tum ciktilar zaten Drive'da: `MyDrive/runs_new/rtdetr_v2_r18vd_from_scratch_restratified_colab/`

- `best.pt` — en iyi `eval_map` snapshot'i (Faster R-CNN ile ayni sema:
  `{epoch, model, cfg, class_names, metric}`)
- `last.pt` — son evaluation snapshot'i
- `training_metrics.xlsx` — her eval'de bir satir
- `plots/` — egitim bitince otomatik uretilen grafikler
- `checkpoints/` — HF Trainer checkpoint'leri (resume kaynagi, son 2 tanesi tutulur)

Oturum kopmussa: son checkpoint'i bul, config hucresinde `RESUME_FROM`'a ver,
config + egitim hucrelerini yeniden calistir.

In [ ]:
# (Opsiyonel) Resume icin son checkpoint'i bul.
!ls -lt {RUNS_NEW}/{CFG['NAME']}*/checkpoints/ 2>/dev/null | head -20